# Digital Twin — Clock School 10-MHz Demonstrator

**Tier 2 · Numerical Clock Networks**

This notebook builds a digital twin of the physical hardware described in the
[10-MHz Demonstrator Lab Guide](../docs/clock-school-demonstrator-guide.md).
Every component in the kit — GPSDO, three OCXOs, oscilloscope, thermistors —
has a numerical counterpart here, driven by the same noise physics that
governs the real devices.

**What you will do:**

| Section | Hardware counterpart | What you learn |
|---|---|---|
| 1. Oscillator models | LBE-1420 GPSDO + 3× OCXO | White, flicker, and random-walk noise |
| 2. Warm-up simulation | OCXO oven transient | Exponential relaxation and residual drift |
| 3. Environment | NTC thermistor + room | Temperature → frequency coupling |
| 4. Phase comparison | Oscilloscope CH1 vs CH2 | Beating, drift, and time-interval error |
| 5. Allan deviation | Stability measurement | σ_y(τ) and noise-type identification |
| 6. Triangular closure | 3-OCXO network | Consistency check for systematic errors |
| 7. Noise budget | Full system | Dominant noise source identification |

**Invariant principle:** *A clock is an operational comparison between periodic
processes that yields information about timing, offset, and relative stability.*
The digital twin preserves this: every measurement is a comparison, never a
reading from a single oscillator in isolation.

---

**Requirements:** `numpy`, `scipy`, `matplotlib`, `allantools`

```
pip install numpy scipy matplotlib allantools
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import allantools
from scipy.signal import welch

plt.rcParams.update({
    "figure.facecolor": "#1a1f2e",
    "axes.facecolor": "#1a1f2e",
    "axes.edgecolor": "#64748b",
    "axes.labelcolor": "#e2e8f0",
    "text.color": "#e2e8f0",
    "xtick.color": "#94a3b8",
    "ytick.color": "#94a3b8",
    "grid.color": "#334155",
    "grid.alpha": 0.5,
    "figure.figsize": (12, 5),
    "font.family": "monospace",
    "font.size": 11,
})

AMBER = "#f59e0b"
GREEN = "#22c55e"
BLUE  = "#3b82f6"
RED   = "#ef4444"
SAND  = "#d4c5a9"

rng = np.random.default_rng(seed=42)
print("Digital twin environment ready.")

## 1. Oscillator Models

The prototype contains four oscillators:

| Device | Model | Nominal frequency | Fractional stability | Noise character |
|---|---|---|---|---|
| GPSDO | LBE-1420 | 10 MHz | ~10⁻¹² | White (short τ), flicker floor (long τ) |
| OCXO A | Supplier 1 | 10 MHz | ~10⁻⁸ – 10⁻⁹ | Flicker + random walk (drift) |
| OCXO B | Supplier 1 | 10 MHz | ~10⁻⁸ – 10⁻⁹ | Flicker + random walk (drift) |
| OCXO C | Supplier 2 | 10 MHz | ~10⁻⁸ – 10⁻⁹ | Flicker + random walk (drift) |

We model each oscillator's fractional frequency deviation $y(t)$ as a sum of
independent noise processes. The GPSDO is our reference — it is not perfect,
but it is ~10⁴× more stable than the OCXOs, so its noise is negligible on the
timescales we care about.

### Noise generation

Three canonical power-law noise types, generated via spectral shaping:

| Noise type | $S_y(f)$ | Allan deviation slope | Physical origin |
|---|---|---|---|
| White frequency | $h_0$ | $\sigma_y \propto \tau^{-1/2}$ | Thermal (Johnson) noise |
| Flicker frequency | $h_{-1}/f$ | $\sigma_y \propto \tau^0$ (floor) | Electronics 1/f noise |
| Random walk frequency | $h_{-2}/f^2$ | $\sigma_y \propto \tau^{+1/2}$ | Environmental drift |

In [ ]:
# ── Noise generators ──

def white_noise(n, h0, rng):
    """White frequency noise: flat PSD at level h0."""
    return np.sqrt(h0) * rng.standard_normal(n)


def flicker_noise(n, h_minus1, rng):
    """Flicker frequency noise via spectral shaping (1/√f filter)."""
    n_fft = n if n % 2 == 0 else n + 1
    freqs = np.fft.rfftfreq(n_fft, d=1.0)
    freqs[0] = freqs[1]  # avoid division by zero at DC
    # Shape white noise in frequency domain
    white = rng.standard_normal(n_fft)
    spectrum = np.fft.rfft(white)
    spectrum *= np.sqrt(h_minus1 / freqs)
    shaped = np.fft.irfft(spectrum, n=n_fft)[:n]
    return shaped


def random_walk_noise(n, h_minus2, rng):
    """Random walk frequency noise: cumulative sum of white noise."""
    white = np.sqrt(h_minus2) * rng.standard_normal(n)
    return np.cumsum(white)


# ── Oscillator class ──

class Oscillator:
    """Digital twin of a 10 MHz oscillator with configurable noise."""

    def __init__(self, name, f0=10e6, h0=0, h_minus1=0, h_minus2=0,
                 freq_offset=0.0):
        self.name = name
        self.f0 = f0
        self.h0 = h0                # white freq noise level (S_y units)
        self.h_minus1 = h_minus1    # flicker freq noise level
        self.h_minus2 = h_minus2    # random walk freq noise level
        self.freq_offset = freq_offset  # static fractional offset

    def fractional_frequency(self, n, dt, rng):
        """Generate n samples of fractional frequency deviation y(t)."""
        y = np.full(n, self.freq_offset)
        if self.h0 > 0:
            y += white_noise(n, self.h0 / dt, rng)
        if self.h_minus1 > 0:
            y += flicker_noise(n, self.h_minus1 / dt, rng)
        if self.h_minus2 > 0:
            y += random_walk_noise(n, self.h_minus2 / dt, rng)
        return y

    def phase(self, n, dt, rng):
        """Generate phase (time error) x(t) in seconds."""
        y = self.fractional_frequency(n, dt, rng)
        return np.cumsum(y) * dt


# ── Define the four oscillators ──
#
# Noise parameter calibration (order-of-magnitude targets):
#
#   White freq:   σ_y(1s) ≈ √(h0/2)           → h0 ~ 2×σ_y²(1s)
#   Flicker freq: σ_y(floor) ≈ √(2 ln2 × h-1) → h-1 ~ σ_y²/(2 ln2)
#   Random walk:  σ_y(τ) ∝ √(h-2 × τ)         → h-2 from long-τ slope
#
# For a typical OCXO:
#   σ_y(1s) ~ 2–5 × 10⁻⁹  → h0 ~ 1–5 × 10⁻¹⁷
#   flicker floor ~ 5 × 10⁻¹⁰ at τ ~ 100s → h-1 ~ 2 × 10⁻¹⁹
#   random walk at τ ~ 10⁴ s ~ 2 × 10⁻⁹ → h-2 ~ 5 × 10⁻²²

# GPSDO: dominant noise is white at ~1e-12 level; negligible on OCXO timescales
gpsdo = Oscillator("LBE-1420 GPSDO", h0=1e-24, h_minus1=1e-26)

# Three OCXOs with slightly different characteristics
# Supplier 1 (two units, similar but not identical)
ocxo_a = Oscillator("OCXO A (Supplier 1)", h0=2e-17, h_minus1=2e-19,
                     h_minus2=5e-22, freq_offset=3e-9)
ocxo_b = Oscillator("OCXO B (Supplier 1)", h0=3e-17, h_minus1=3e-19,
                     h_minus2=8e-22, freq_offset=-2e-9)
# Supplier 2 (different crystal cut, slightly different drift)
ocxo_c = Oscillator("OCXO C (Supplier 2)", h0=1.5e-17, h_minus1=4e-19,
                     h_minus2=1e-21, freq_offset=5e-9)

oscillators = [gpsdo, ocxo_a, ocxo_b, ocxo_c]
print("Oscillators defined:")
for osc in oscillators:
    print(f"  {osc.name}: h0={osc.h0:.0e}, h-1={osc.h_minus1:.0e}, "
          f"h-2={osc.h_minus2:.0e}, offset={osc.freq_offset:+.1e}")

## 2. OCXO Warm-Up Simulation

When you power on an OCXO, the internal oven heats the crystal to its
turnover temperature. During warm-up:

- The frequency drifts **fast and visibly** (you can watch this on the oscilloscope)
- The drift follows an approximately **exponential relaxation** toward the
  steady-state frequency
- After ~20–30 minutes, the oven stabilises and only slow residual drift remains

This is the first thing a student sees when connecting the prototype.
The digital twin reproduces it.

In [ ]:
# ── Warm-up model ──

def ocxo_warmup(t, delta_y0=-2e-6, tau_oven=300.0, drift_rate=1e-11):
    """
    OCXO warm-up transient as fractional frequency deviation.

    Parameters
    ----------
    t : array, seconds since power-on
    delta_y0 : initial fractional offset at cold start (~-2 ppm typical)
    tau_oven : oven time constant in seconds (~5 min)
    drift_rate : residual linear drift after stabilisation (per second)
    """
    return delta_y0 * np.exp(-t / tau_oven) + drift_rate * t


# Simulate 2 hours of warm-up for the three OCXOs
dt_warmup = 1.0  # 1 second sample interval
t_warmup = np.arange(0, 7200, dt_warmup)
n_warmup = len(t_warmup)

# Each OCXO has slightly different warm-up characteristics
warmup_params = {
    "OCXO A": {"delta_y0": -1.8e-6, "tau_oven": 280, "drift_rate": 8e-12},
    "OCXO B": {"delta_y0": -2.2e-6, "tau_oven": 320, "drift_rate": 1.2e-11},
    "OCXO C": {"delta_y0": -1.5e-6, "tau_oven": 250, "drift_rate": 1.5e-11},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, params in warmup_params.items():
    y_warmup = ocxo_warmup(t_warmup, **params)
    # Add noise on top of the deterministic warm-up
    y_warmup += flicker_noise(n_warmup, 5e-16, rng)
    color = {"OCXO A": AMBER, "OCXO B": GREEN, "OCXO C": BLUE}[name]

    ax1.plot(t_warmup / 60, y_warmup * 1e6, color=color, alpha=0.8,
             label=name, linewidth=0.8)
    # Zoom into the stabilised region (last hour)
    mask = t_warmup >= 3600
    ax2.plot(t_warmup[mask] / 60, y_warmup[mask] * 1e9, color=color,
             alpha=0.8, label=name, linewidth=0.8)

ax1.set_xlabel("Time since power-on (minutes)")
ax1.set_ylabel("Fractional frequency offset (ppm)")
ax1.set_title("OCXO Warm-Up Transient")
ax1.legend(fontsize=9)
ax1.grid(True)
ax1.axhline(0, color=SAND, linewidth=0.5, linestyle="--")

ax2.set_xlabel("Time since power-on (minutes)")
ax2.set_ylabel("Fractional frequency offset (ppb)")
ax2.set_title("After Stabilisation (zoomed)")
ax2.legend(fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.show()

print("Observe: fast drift during warm-up → slow residual drift after oven stabilises.")

## 3. Temperature Environment

The prototype includes five NTC 10K thermistors. In the real kit, you tape
one to each OCXO body and place others around the room. Temperature changes
couple to frequency through the crystal's temperature coefficient.

The digital twin generates a realistic 24-hour temperature profile:
a diurnal cycle (day/night swing) plus short-term fluctuations (air
conditioning, doors opening, sunlight patches).

The Steinhart-Hart equation converts the thermistor resistance to temperature,
exactly as you would with the FNIRSI 2C53P multimeter in resistance mode.

In [ ]:
# ── Temperature environment simulation ──

dt_env = 1.0       # 1 second sample interval
t_env = np.arange(0, 24 * 3600, dt_env)  # 24 hours
n_env = len(t_env)

# Room temperature: diurnal cycle + short-term noise
T_mean = 28.0       # mean room temperature (Cameroon indoor, °C)
T_diurnal = 4.0     # peak-to-peak diurnal swing (°C)
T_room = (T_mean
          + (T_diurnal / 2) * np.sin(2 * np.pi * t_env / 86400 - np.pi / 2)
          + 0.3 * rng.standard_normal(n_env))  # short-term fluctuations

# OCXO body temperature: oven holds ~80°C, but residual coupling to room
# The oven suppresses room temperature variations by a factor of ~100-1000
T_oven_setpoint = 80.0
oven_suppression = 500.0  # oven attenuation factor
T_ocxo = T_oven_setpoint + (T_room - T_mean) / oven_suppression

# NTC thermistor model: Steinhart-Hart for NTC 10K 3950
B_VALUE = 3950.0
R_25 = 10_000.0  # resistance at 25°C

def ntc_resistance(T_celsius):
    """NTC 10K 3950 resistance from temperature."""
    T_k = T_celsius + 273.15
    return R_25 * np.exp(B_VALUE * (1.0 / T_k - 1.0 / 298.15))

def ntc_temperature(R):
    """Temperature from NTC resistance (inverse Steinhart-Hart, B-value)."""
    return 1.0 / (1.0 / 298.15 + np.log(R / R_25) / B_VALUE) - 273.15

R_room = ntc_resistance(T_room)
R_ocxo = ntc_resistance(T_ocxo)

# ── Plot ──
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Room temperature
axes[0, 0].plot(t_env / 3600, T_room, color=AMBER, linewidth=0.5, alpha=0.7)
axes[0, 0].set_ylabel("Temperature (°C)")
axes[0, 0].set_title("Room Temperature (NTC on wall)")
axes[0, 0].grid(True)

# NTC resistance
axes[0, 1].plot(t_env / 3600, R_room / 1000, color=GREEN, linewidth=0.5, alpha=0.7)
axes[0, 1].set_ylabel("Resistance (kΩ)")
axes[0, 1].set_title("NTC Thermistor Reading (multimeter)")
axes[0, 1].grid(True)

# OCXO body temperature
axes[1, 0].plot(t_env / 3600, (T_ocxo - T_oven_setpoint) * 1000,
                color=BLUE, linewidth=0.5, alpha=0.7)
axes[1, 0].set_ylabel("ΔT from setpoint (m°C)")
axes[1, 0].set_title("OCXO Crystal Temperature Residual")
axes[1, 0].set_xlabel("Time (hours)")
axes[1, 0].grid(True)

# Verify NTC inversion round-trip
T_reconstructed = ntc_temperature(R_room)
axes[1, 1].plot(t_env / 3600, (T_reconstructed - T_room) * 1e6,
                color=RED, linewidth=0.5, alpha=0.7)
axes[1, 1].set_ylabel("Residual (µ°C)")
axes[1, 1].set_title("NTC Round-Trip Error (should be ~0)")
axes[1, 1].set_xlabel("Time (hours)")
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print(f"Room: {T_room.mean():.1f} ± {T_room.std():.2f} °C")
print(f"OCXO crystal residual: {(T_ocxo - T_oven_setpoint).std() * 1000:.2f} m°C RMS")

## 4. Phase Comparison — Simulated Oscilloscope

This is the core Clock School experiment: **making stability visible.**

On the real oscilloscope (FNIRSI 2C53P), you connect the GPSDO to channel 1
and an OCXO to channel 2, both at 10 MHz. At first the signals look identical.
Over minutes, the OCXO drifts in phase — the waveform slides sideways
relative to the reference.

The digital twin reproduces this by computing the phase difference
$\Delta\phi(t) = \phi_\text{OCXO}(t) - \phi_\text{GPSDO}(t)$
and showing both the "oscilloscope" view (fast timescale) and the
time-interval error (slow timescale).

In [ ]:
# ── 24-hour comparison: GPSDO vs three OCXOs ──

dt = 1.0  # 1 second gate time (matches real counter measurement)
t = np.arange(0, 24 * 3600, dt)
n = len(t)

# Temperature-coupled fractional frequency for each OCXO.
# The oven suppresses room temperature variations by ~500×.
# Residual crystal temperature excursion is ~few m°C peak.
# Typical OCXO temp coefficient near turnover: ~1e-9 per °C of crystal temp.
temp_coeff_per_degC = 1e-9  # Δy per °C of crystal temperature change
delta_T_crystal_degC = (T_room - T_mean) / oven_suppression  # ~m°C scale, in °C

# Generate frequency data for each oscillator
y_gpsdo = gpsdo.fractional_frequency(n, dt, rng)

ocxo_list = [ocxo_a, ocxo_b, ocxo_c]
y_ocxo = {}
x_ocxo = {}  # phase (time-interval error)

for osc in ocxo_list:
    y = osc.fractional_frequency(n, dt, rng)
    # Add temperature coupling (both variables in consistent units: °C)
    y += temp_coeff_per_degC * delta_T_crystal_degC
    y_ocxo[osc.name] = y
    # Phase = cumulative sum of (OCXO - GPSDO) frequency × dt
    x_ocxo[osc.name] = np.cumsum(y - y_gpsdo) * dt

# ── Simulated oscilloscope: snapshot at three different times ──

f0 = 10e6
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

snapshots = [
    (60, "t = 1 min (just after lock)"),
    (3600, "t = 1 hour"),
    (20 * 3600, "t = 20 hours"),
]

for col, (t_snap, title) in enumerate(snapshots):
    idx = int(t_snap / dt)
    # Phase difference at this moment (in radians of 10 MHz)
    delta_x = x_ocxo[ocxo_a.name][idx]  # time error in seconds
    delta_phi = 2 * np.pi * f0 * delta_x  # phase in radians

    # Render ~3 cycles of 10 MHz (300 ns window)
    t_scope = np.linspace(0, 300e-9, 600)
    ch1 = np.sin(2 * np.pi * f0 * t_scope)  # GPSDO (reference)
    ch2 = np.sin(2 * np.pi * f0 * t_scope + delta_phi)  # OCXO A

    axes[0, col].plot(t_scope * 1e9, ch1, color=AMBER, linewidth=1.2,
                      label="CH1: GPSDO")
    axes[0, col].plot(t_scope * 1e9, ch2, color=GREEN, linewidth=1.2,
                      label="CH2: OCXO A", alpha=0.8)
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].set_ylim(-1.4, 1.4)
    axes[0, col].set_xlabel("Time (ns)")
    axes[0, col].grid(True)
    if col == 0:
        axes[0, col].set_ylabel("Amplitude (norm.)")
        axes[0, col].legend(fontsize=8, loc="upper right")

# Time-interval error (full 24 hours)
colors = [AMBER, GREEN, BLUE]
for i, osc in enumerate(ocxo_list):
    axes[1, 0].plot(t / 3600, x_ocxo[osc.name] * 1e6, color=colors[i],
                    linewidth=0.6, alpha=0.8, label=osc.name.split(" (")[0])
axes[1, 0].set_xlabel("Time (hours)")
axes[1, 0].set_ylabel("Time-interval error (µs)")
axes[1, 0].set_title("Phase Drift: OCXO vs GPSDO")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True)

# Fractional frequency (1-hour segment)
seg = slice(3600, 7200)
for i, osc in enumerate(ocxo_list):
    axes[1, 1].plot(t[seg] / 3600, y_ocxo[osc.name][seg] * 1e9,
                    color=colors[i], linewidth=0.4, alpha=0.7,
                    label=osc.name.split(" (")[0])
axes[1, 1].set_xlabel("Time (hours)")
axes[1, 1].set_ylabel("Fractional frequency (ppb)")
axes[1, 1].set_title("Fractional Frequency (1-hour window)")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True)

# Temperature correlation
ax_twin = axes[1, 2]
ax_twin.plot(t / 3600, T_room, color=RED, linewidth=0.4, alpha=0.6)
ax_twin.set_xlabel("Time (hours)")
ax_twin.set_ylabel("Room temperature (°C)", color=RED)
ax_twin.set_title("Temperature vs OCXO A Frequency")
ax_twin.grid(True)
ax2 = ax_twin.twinx()
ax2.plot(t / 3600, y_ocxo[ocxo_a.name] * 1e9, color=AMBER,
         linewidth=0.4, alpha=0.6)
ax2.set_ylabel("Fractional frequency (ppb)", color=AMBER)

plt.tight_layout()
plt.show()

# Report drift in appropriate units
for osc in ocxo_list:
    drift_ns = x_ocxo[osc.name][-1] * 1e9
    label = osc.name.split(" (")[0]
    if abs(drift_ns) > 1000:
        print(f"{label} phase drift after 24h: {drift_ns/1000:.1f} µs")
    else:
        print(f"{label} phase drift after 24h: {drift_ns:.1f} ns")

## 5. Allan Deviation

The Allan deviation $\sigma_y(\tau)$ is the standard tool for characterising
oscillator stability. It tells you how stable a clock is *as a function of
averaging time* $\tau$.

The slope of $\sigma_y(\tau)$ on a log-log plot identifies the dominant noise type:

| Slope | Noise type | Meaning |
|---|---|---|
| $\tau^{-1}$ | White phase | Measurement noise |
| $\tau^{-1/2}$ | White frequency | Thermal noise floor |
| $\tau^{0}$ (flat) | Flicker frequency | Electronics 1/f noise |
| $\tau^{+1/2}$ | Random walk frequency | Environmental drift |
| $\tau^{+1}$ | Frequency drift | Ageing, linear drift |

We compute overlapping Allan deviation (OADEV) for each OCXO, measured
against the GPSDO — exactly as you would with real counter data.

In [ ]:
# ── Allan deviation for each OCXO vs GPSDO ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

labels_short = ["OCXO A", "OCXO B", "OCXO C"]

for i, osc in enumerate(ocxo_list):
    # Fractional frequency difference: OCXO - GPSDO
    y_diff = y_ocxo[osc.name] - y_gpsdo

    # Overlapping Allan deviation
    taus_requested = np.logspace(0, 4, 60)  # 1 s to 10000 s
    (taus, adevs, errors, ns) = allantools.oadev(
        y_diff, rate=1.0 / dt, data_type="freq", taus=taus_requested
    )

    ax1.loglog(taus, adevs, color=colors[i], linewidth=1.5,
               label=labels_short[i], alpha=0.9)
    ax1.fill_between(taus, adevs - errors, adevs + errors,
                     color=colors[i], alpha=0.15)

# Reference slopes (anchored to typical OCXO levels)
tau_ref = np.array([1, 1e4])
ax1.loglog(tau_ref, 3e-9 * np.sqrt(1 / tau_ref), "--", color=SAND,
           linewidth=0.8, alpha=0.6, label=r"$\tau^{-1/2}$ (white freq)")
ax1.loglog(tau_ref, 5e-10 * np.ones_like(tau_ref), ":", color=SAND,
           linewidth=0.8, alpha=0.6, label=r"$\tau^{0}$ (flicker floor)")
ax1.loglog(tau_ref, 5e-11 * np.sqrt(tau_ref), "-.", color=SAND,
           linewidth=0.8, alpha=0.6, label=r"$\tau^{+1/2}$ (random walk)")

ax1.set_xlabel(r"Averaging time $\tau$ (s)")
ax1.set_ylabel(r"Overlapping Allan deviation $\sigma_y(\tau)$")
ax1.set_title("Allan Deviation: Each OCXO vs GPSDO")
ax1.legend(fontsize=8, loc="upper right")
ax1.grid(True, which="both", alpha=0.3)

# ── Power spectral density ──
for i, osc in enumerate(ocxo_list):
    y_diff = y_ocxo[osc.name] - y_gpsdo
    f_psd, psd = welch(y_diff, fs=1.0/dt, nperseg=min(8192, n//4))
    ax2.loglog(f_psd[1:], psd[1:], color=colors[i], linewidth=0.8,
               alpha=0.8, label=labels_short[i])

ax2.set_xlabel("Fourier frequency (Hz)")
ax2.set_ylabel(r"$S_y(f)$ (1/Hz)")
ax2.set_title("Power Spectral Density of Frequency Fluctuations")
ax2.legend(fontsize=8)
ax2.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

print("Read the Allan deviation plot:")
print("  - Short τ: slope ~ -1/2 → white frequency noise dominates")
print("  - Mid τ: flattening → flicker floor")
print("  - Long τ: slope ~ +1/2 → random walk (drift) takes over")

## 6. Triangular Closure

The prototype includes three OCXOs. With three clocks, you can form a
**triangular comparison network**: measure all three pairwise frequency
differences and check whether they sum to zero.

$$y_{AB} + y_{BC} + y_{CA} = 0$$

If the closure residual is significantly non-zero, something is wrong:
a systematic error, a measurement artefact, or a noise source you
haven't modelled.

This is the same logic used by BIPM to validate international clock
comparisons — here applied to three modules on a bench in Cameroon.

In [ ]:
# ── Triangular closure ──

# Pairwise frequency differences
y_ab = y_ocxo[ocxo_a.name] - y_ocxo[ocxo_b.name]
y_bc = y_ocxo[ocxo_b.name] - y_ocxo[ocxo_c.name]
y_ca = y_ocxo[ocxo_c.name] - y_ocxo[ocxo_a.name]

# Closure residual: should be exactly zero for ideal measurements
closure = y_ab + y_bc + y_ca

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Pairwise differences
pairs = [("A − B", y_ab, AMBER), ("B − C", y_bc, GREEN), ("C − A", y_ca, BLUE)]
for ax, (label, y_pair, color) in zip(axes[:2], pairs[:2]):
    ax.plot(t / 3600, y_pair * 1e9, color=color, linewidth=0.4, alpha=0.7)
    ax.set_xlabel("Time (hours)")
    ax.set_ylabel("Δy (ppb)")
    ax.set_title(f"Pairwise: {label}")
    ax.grid(True)

# Third pair + closure
axes[2].plot(t / 3600, y_ca * 1e9, color=BLUE, linewidth=0.4, alpha=0.5,
             label="C − A")
axes[2].plot(t / 3600, closure * 1e15, color=RED, linewidth=1.0,
             alpha=0.9, label=f"Closure (×10⁶)")
axes[2].set_xlabel("Time (hours)")
axes[2].set_ylabel("Δy (ppb) / Closure (fHz/Hz)")
axes[2].set_title("C − A + Closure Residual")
axes[2].legend(fontsize=8)
axes[2].grid(True)

plt.tight_layout()
plt.show()

closure_rms = np.sqrt(np.mean(closure**2))
print(f"Closure residual RMS: {closure_rms:.2e}")
print(f"This is {'consistent with numerical precision' if closure_rms < 1e-15 else 'non-zero — investigate!'}.")
print()
print("In the real prototype, a non-zero closure indicates:")
print("  - Cable-dependent systematic errors (impedance mismatch, reflections)")
print("  - Oscilloscope trigger jitter affecting different channels differently")
print("  - Thermal gradients across the bench")

# ── Allan deviation of pairwise comparisons ──
fig, ax = plt.subplots(figsize=(8, 5))

for label, y_pair, color in pairs:
    taus_req = np.logspace(0, 4, 50)
    (taus, adevs, errors, ns) = allantools.oadev(
        y_pair, rate=1.0/dt, data_type="freq", taus=taus_req
    )
    ax.loglog(taus, adevs, color=color, linewidth=1.5, label=label, alpha=0.9)

ax.set_xlabel(r"Averaging time $\tau$ (s)")
ax.set_ylabel(r"$\sigma_y(\tau)$")
ax.set_title("Allan Deviation: Pairwise OCXO Comparisons")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

print("Compare these to the GPSDO-referenced curves above.")
print("Pairwise OCXO comparisons are noisier (both oscillators contribute noise).")

## 7. Noise Budget

A noise budget answers the question: **which noise source dominates the
measurement, and at what timescale?**

For each OCXO we decompose the total Allan deviation into contributions
from white noise, flicker noise, random-walk noise, and temperature coupling.
This is the same analysis a clock engineer would perform to decide whether
shielding, temperature control, or better electronics would improve the system.

The noise budget also computes the **causal-geometry parameter**
$\eta(\tau) = L / (c \cdot \tau)$ for the bench setup, confirming that
propagation delay is negligible at Tier 1 scales.

In [ ]:
# ── Noise budget: decompose Allan deviation by source ──

# Generate isolated noise components for OCXO A
n_budget = 24 * 3600  # 24 hours at 1 Hz
rng_budget = np.random.default_rng(seed=100)

y_white_only = white_noise(n_budget, ocxo_a.h0 / dt, rng_budget)
y_flicker_only = flicker_noise(n_budget, ocxo_a.h_minus1 / dt, rng_budget)
y_rw_only = random_walk_noise(n_budget, ocxo_a.h_minus2 / dt, rng_budget)
y_temp_only = temp_coeff_per_degC * delta_T_crystal_degC  # temperature coupling

components = [
    ("White frequency",    y_white_only,   AMBER),
    ("Flicker frequency",  y_flicker_only, GREEN),
    ("Random walk",        y_rw_only,      BLUE),
    ("Temperature coupling", y_temp_only,  RED),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

taus_req = np.logspace(0, 4, 50)

for label, y_comp, color in components:
    (taus, adevs, errors, ns) = allantools.oadev(
        y_comp, rate=1.0/dt, data_type="freq", taus=taus_req
    )
    ax1.loglog(taus, adevs, color=color, linewidth=1.5, label=label, alpha=0.9)

# Total (all components summed)
y_total = y_white_only + y_flicker_only + y_rw_only + y_temp_only
(taus_tot, adevs_tot, _, _) = allantools.oadev(
    y_total, rate=1.0/dt, data_type="freq", taus=taus_req
)
ax1.loglog(taus_tot, adevs_tot, color=SAND, linewidth=2.0,
           label="Total (combined)", linestyle="--")

ax1.set_xlabel(r"Averaging time $\tau$ (s)")
ax1.set_ylabel(r"$\sigma_y(\tau)$")
ax1.set_title("Noise Budget: OCXO A vs GPSDO")
ax1.legend(fontsize=8)
ax1.grid(True, which="both", alpha=0.3)

# ── η(τ) causal-geometry parameter ──
c = 3e8  # speed of light (m/s)
L_bench = 1.0  # cable length on bench (metres)
L_campus = 100.0  # hypothetical campus deployment
L_satellite = 20_200e3  # GPS orbit

taus_eta = np.logspace(-1, 5, 200)
for L, label, color, ls in [
    (L_bench,     "Bench (1 m)",       AMBER, "-"),
    (L_campus,    "Campus (100 m)",    GREEN, "--"),
    (L_satellite, "GPS orbit (20 Mm)", BLUE,  ":"),
]:
    eta = L / (c * taus_eta)
    ax2.loglog(taus_eta, eta, color=color, linewidth=1.5, linestyle=ls,
               label=label)

ax2.axhline(1.0, color=RED, linewidth=0.8, linestyle="--", alpha=0.5)
ax2.text(1e4, 1.3, "causal limit (η = 1)", color=RED, fontsize=9, alpha=0.7)
ax2.axhline(0.01, color=SAND, linewidth=0.5, linestyle=":", alpha=0.4)
ax2.text(1e4, 0.013, "propagation becomes relevant", color=SAND,
         fontsize=8, alpha=0.5)

ax2.set_xlabel(r"Averaging time $\tau$ (s)")
ax2.set_ylabel(r"$\eta(\tau) = L / (c \cdot \tau)$")
ax2.set_title("Causal-Geometry Parameter")
ax2.legend(fontsize=9)
ax2.grid(True, which="both", alpha=0.3)
ax2.set_ylim(1e-12, 10)

plt.tight_layout()
plt.show()

print("Noise budget interpretation:")
print("  - Short τ (<10 s): white frequency noise dominates → average longer")
print("  - Mid τ (10–1000 s): flicker floor → averaging gives no further gain")
print("  - Long τ (>1000 s): random walk + temperature drift → stabilise environment")
print()
eta_bench_1s = L_bench / (c * 1.0)
print(f"η(τ=1s) on the bench: {eta_bench_1s:.2e}")
print("  → Propagation delay is completely negligible at Tier 1 scales.")

## Summary

This notebook is a digital twin of the physical kit described in the
[10-MHz Demonstrator Lab Guide](../docs/clock-school-demonstrator-guide.md).
Every section corresponds to something you can do — or will see — with the
real hardware:

| Digital twin | Physical prototype |
|---|---|
| `Oscillator` class with noise parameters | LBE-1420 GPSDO + 3 OCXOs |
| `ocxo_warmup()` exponential transient | Power on an OCXO, watch oscilloscope |
| `ntc_resistance()` / `ntc_temperature()` | Multimeter + NTC thermistor on OCXO body |
| Phase-difference time series | Two-channel oscilloscope, CH1 vs CH2 |
| `allantools.oadev()` | Frequency counter + stability analysis software |
| Triangular closure residual | Three-OCXO comparison network |
| Noise budget decomposition | Engineer's design decision tool |

**What to do next:**

1. **Adjust the noise parameters** to match your real measurements (Exercise 1.3
   from the hardware spec: hand-warm the OCXO, measure the temperature coefficient)
2. **Import real data** from the oscilloscope CSV export and compute Allan deviation
3. **Compare** the digital twin's predictions to your measured stability — where do
   they agree? Where do they disagree? What does the disagreement tell you?
4. **Extend** the network: add the GPSDO as a fourth node, compute all six pairwise
   comparisons, test closure on all four triangles

---

*Clock School — What Is a Clock? · Tier 2 Digital Twin · v0.3*
*CC BY-SA 4.0 · Ulrich Warring and contributors*